# Molecular dynamics of a Lennard-Jones fluid

The function below follows `n` particles of unit mass in a cubic box with periodic edges. Two
particles a distance `r` apart have the Lennard-Jones potential energy `4 * (r**-12 - r**-6)`,
in units in which both the depth of the potential and the particle diameter are 1. Only pairs
closer than the cut-off distance interact, and each particle sees only the nearest periodic copy
of every other particle. The forces move the particles by the velocity Verlet method, which
alternates half steps of the velocities with whole steps of the positions. After each step the
function records the potential energy per particle and the pressure. The pressure is the sum of
the kinetic term and the virial, which is the sum over interacting pairs of the distance times
the force. The function returns the averages over the second half of the steps. Every particle
is compared with every other at every step, so the work grows as the square of the number of
particles. The demonstration places two particles at rest at the distance where the force
between them vanishes, and prints the potential energy per particle and the pressure.

In [ ]:
import numpy as np


def interactions(positions, box, cutoff):
    """Forces on the particles, the potential energy and the virial, with periodic edges.

    Two particles closer than `cutoff`, measured between the nearest periodic copies, have the
    potential energy 4 * (r**-12 - r**-6) at a distance r. Farther apart they do not interact.
    The virial is the sum over interacting pairs of the distance between them times the force
    between them. The function handles a block of rows of the table of pairs at a time.
    """
    n = len(positions)
    forces = np.zeros_like(positions)
    potential = virial = 0.0
    for start in range(0, n, 256):
        rows = slice(start, start + 256)
        delta = positions[rows, None, :] - positions[None, :, :]
        delta -= box * np.round(delta / box)  # the nearest periodic copy
        r2 = np.sum(delta * delta, axis=2)
        inside = r2 < cutoff * cutoff
        inside[np.arange(len(r2)), np.arange(start, start + len(r2))] = False  # not itself
        inv_r2 = np.divide(1.0, r2, out=np.zeros_like(r2), where=inside)
        inv_r6 = inv_r2**3
        potential += 2.0 * np.sum(inv_r6 * (inv_r6 - 1.0))  # every pair appears twice
        slope = 24.0 * inv_r2 * inv_r6 * (2.0 * inv_r6 - 1.0)  # the force divided by the distance
        forces[rows] = np.einsum("ij,ijk->ik", slope, delta)
        virial += 0.5 * np.sum(slope * r2)
    return forces, potential, virial


def md(positions, velocities, box, dt, steps, cutoff):
    """Mean potential energy per particle and mean pressure over the second half of the steps.

    The positions are not wrapped back into the box as the particles move. The motion is
    chaotic, so a routine agrees with this function only to within statistical error.
    """
    n, d = positions.shape
    positions, velocities = positions.copy(), velocities.copy()
    forces, potential, virial = interactions(positions, box, cutoff)
    potential_sum = pressure_sum = 0.0
    for step in range(steps):
        velocities += 0.5 * dt * forces
        positions += dt * velocities
        forces, potential, virial = interactions(positions, box, cutoff)
        velocities += 0.5 * dt * forces
        if step >= steps // 2:
            kinetic = 0.5 * np.sum(velocities * velocities)
            potential_sum += potential / n
            pressure_sum += (2.0 * kinetic + virial) / (d * box**d)
    count = steps - steps // 2
    return potential_sum / count, pressure_sum / count

In [ ]:
print("potential energy per particle and pressure: %.3f %.3f"
      % md(np.array([[0.0, 0.0, 0.0], [2 ** (1 / 6), 0.0, 0.0]]), np.zeros((2, 3)),
           box=10.0, dt=0.005, steps=100, cutoff=2.5))

The declaration below tells the harness how to call the function and how to check a routine
against it. The positions and velocities are tables with a row for each of `n` particles and a
column for each of the three directions, and `n` becomes the last parameter of the C
routine. `make_inputs` builds a cubic lattice of about `size` particles at a number density of
0.8 and moves each particle a little at random. It gives each particle a random velocity,
removes the average velocity, and scales the velocities so that the temperature is exactly 1.
Finally it leaves a few particles exactly on their lattice sites and sets a few at rest.

The motion is chaotic, so in a long run two correct routines would agree on the averages only to
within their statistical uncertainty. The tolerances allow for that uncertainty. They are
absolute, and each is five times the standard deviation of the function's own result over 60
seeds, for the smallest system that the tests use. In the run of 1000 steps used here, a plain C
routine agrees with the function to within 2e-7, far inside the tolerances.

In [ ]:
def make_inputs(size, rng):
    cells = max(1, round(size ** (1 / 3)))
    n = cells**3
    box = (n / 0.8) ** (1 / 3)
    spacing = box / cells
    sites = np.indices((cells, cells, cells)).reshape(3, n).T
    jitter = rng.uniform(-0.05, 0.05, (n, 3))
    jitter[rng.random(n) < 0.02] = 0.0
    velocities = rng.normal(0.0, 1.0, (n, 3))
    velocities -= velocities.mean(axis=0)
    if np.any(velocities):
        velocities *= np.sqrt(3 * n / np.sum(velocities**2))
    velocities[rng.random(n) < 0.02] = 0.0
    return {"positions": (sites + 0.5 + jitter) * spacing, "velocities": velocities,
            "box": box, "dt": 0.005, "steps": 1000, "cutoff": min(2.5, box / 2)}


ROUTINE = {
    "function": md,
    "make_inputs": make_inputs,
    "arguments": {"positions": "float64[n, 3]", "velocities": "float64[n, 3]", "box": "float64",
                  "dt": "float64", "steps": "int64", "cutoff": "float64"},
    "results": {"potential": "float64", "pressure": "float64"},
    "test_sizes": [125, 216],
    "benchmark_size": 343,
    "tolerance": {"potential": ("absolute", 0.1), "pressure": ("absolute", 0.5)},
}